In [1]:
import sqlite3
import pandas as pd
 
conn = sqlite3.connect('hospital.db')
df   = pd.read_sql('SELECT * FROM patients', conn)
print(df)


   id   name  age      condition        date
0   1  Alice   34       Diabetes  2024-01-10
1   2    Bob   55  Heart Disease  2024-01-15
2   3  Chidi   28        Malaria  2024-02-01
3   4  Amara   42       Diabetes  2024-02-14
4   5  James   61  Heart Disease  2024-03-03


In [2]:
import sqlite3
import pandas as pd

# ── Setup ──────────────────────────────────────────────────
conn = sqlite3.connect('hospital.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS patients (
        id INTEGER PRIMARY KEY, name TEXT,
        age INTEGER, condition TEXT, date TEXT
    )
''')
cursor.execute("DELETE FROM patients")
cursor.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", [
    (1, 'Alice', 34, 'Diabetes',      '2024-01-10'),
    (2, 'Bob',   55, 'Heart Disease', '2024-01-15'),
    (3, 'Chidi', 28, 'Malaria',       '2024-02-01'),
    (4, 'Amara', 42, 'Diabetes',      '2024-02-14'),
    (5, 'James', 61, 'Heart Disease', '2024-03-03'),
])
conn.commit()

# ── Q1: Most common condition ───────────────────────────────
q1 = pd.read_sql('''
    SELECT condition, COUNT(*) as total
    FROM patients
    GROUP BY condition
    ORDER BY total DESC
    LIMIT 1
''', conn)
condition = q1['condition'][0]
total = q1['total'][0]
print(f"Q1: The most common condition is {condition}, affecting {total} out of 5 patients.")

# ── Q2: Average patient age ─────────────────────────────────
q2 = pd.read_sql('SELECT ROUND(AVG(age), 1) as avg_age FROM patients', conn)
avg_age = q2['avg_age'][0]
print(f"Q2: The average age of patients is {avg_age} years old.")

# ── Q3: Oldest and youngest patient ────────────────────────
q3 = pd.read_sql('''
    SELECT name, age FROM patients
    WHERE age = (SELECT MAX(age) FROM patients)
       OR age = (SELECT MIN(age) FROM patients)
    ORDER BY age DESC
''', conn)
oldest = q3.iloc[0]
youngest = q3.iloc[-1]
print(f"Q3: The oldest patient is {oldest['name']} at {int(oldest['age'])} years old. "
      f"The youngest is {youngest['name']} at {int(youngest['age'])} years old.")

# ── Q4: Which condition has the highest average age ─────────
q4 = pd.read_sql('''
    SELECT condition, ROUND(AVG(age), 1) as avg_age
    FROM patients
    GROUP BY condition
    ORDER BY avg_age DESC
    LIMIT 1
''', conn)
print(f"Q4: Patients with {q4['condition'][0]} have the highest average age "
      f"at {q4['avg_age'][0]} years, suggesting it is more common in older patients.")

# ── Q5: Busiest admission month ─────────────────────────────
q5 = pd.read_sql('''
    SELECT SUBSTR(date, 1, 7) as month, COUNT(*) as admissions
    FROM patients
    GROUP BY month
    ORDER BY admissions DESC
    LIMIT 1
''', conn)
print(f"Q5: The busiest admission month was {q5['month'][0]} "
      f"with {q5['admissions'][0]} patient(s) admitted.")

conn.close()


Q1: The most common condition is Heart Disease, affecting 2 out of 5 patients.
Q2: The average age of patients is 44.0 years old.
Q3: The oldest patient is James at 61 years old. The youngest is Chidi at 28 years old.
Q4: Patients with Heart Disease have the highest average age at 58.0 years, suggesting it is more common in older patients.
Q5: The busiest admission month was 2024-02 with 2 patient(s) admitted.
